(15)=
# Chapter 15: Numerical Integration and Differentiation

**Topics Covered:**
- Numerical differentiation with `np.gradient`
- Trapezoidal rule: `scipy.integrate.trapezoid`
- General-purpose quadrature: `scipy.integrate.quad`
- Cumulative integration: `scipy.integrate.cumulative_trapezoid`
- ChE application: PFR design and the Levenspiel integral

## Motivation: Two Problems You Can't Solve by Hand

**Problem 1 — How much heat does it take?**

The heat capacity of nitrogen gas is not constant — it varies with temperature. To heat 1 mol of N₂ from 300 K to 1200 K, you need:

$$\Delta H = \int_{300}^{1200} C_p(T) \, dT$$

where $C_p(T)$ follows the NIST Shomate equation. You could try to integrate this polynomial by hand, but what if your $C_p$ data comes from a table of measurements rather than a formula?

**Problem 2 — What is the reaction rate at each time step?**

In a batch reactor experiment, you measure concentration $C_A$ at discrete time points. The reaction rate is:
$$r_A = -\frac{dC_A}{dt}$$

But you don't have a formula for $C_A(t)$ — only a table of numbers. How do you compute a derivative from data?

Both problems need **numerical** methods. Let's look at the tools Python gives us.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad, trapezoid, cumulative_trapezoid

(15.1)=
## 15.1 The Core Idea: Integration and Differentiation Geometrically

Before writing any code, let's make the geometry clear.

- **Integration** = the area under a curve between two limits
- **Differentiation** = the slope of a curve at a point

When you have a smooth analytic function, calculus gives exact answers. When you have **discrete data** or a **complicated expression**, you use numerical approximations instead.

In [ ]:
x = np.linspace(0, 4, 300)
f = x * np.exp(-x)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: integration = area under curve
ax = axes[0]
ax.plot(x, f, 'steelblue', linewidth=2.5)
mask = (x >= 0.5) & (x <= 3.0)
ax.fill_between(x[mask], f[mask], alpha=0.3, color='steelblue', label=r'Area = $\int_{0.5}^{3} f(x)\,dx$')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Integration = area under the curve')
ax.legend(fontsize=10)

# Right: differentiation = slope at a point
ax = axes[1]
ax.plot(x, f, 'steelblue', linewidth=2.5)
x0 = 1.0
f0 = x0 * np.exp(-x0)
dfdx0 = np.exp(-x0) - x0 * np.exp(-x0)   # exact derivative: e^{-x}(1-x)
x_tan = np.linspace(0.2, 1.8, 50)
ax.plot(x_tan, f0 + dfdx0 * (x_tan - x0), 'r--', linewidth=2, label=f"Tangent at x=1\nslope = {dfdx0:.3f}")
ax.scatter([x0], [f0], color='red', s=60, zorder=5)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Differentiation = slope at a point')
ax.legend(fontsize=10)

plt.suptitle(r'$f(x) = x\,e^{-x}$', fontsize=13)
plt.tight_layout()
plt.show()

(15.2)=
## 15.2 Numerical Differentiation: `np.gradient`

When you have a table of $(x, y)$ values and want $dy/dx$, `np.gradient` is the tool.

It uses the **central difference formula** at interior points:
$$f'(x_i) \approx \frac{f(x_{i+1}) - f(x_{i-1})}{x_{i+1} - x_{i-1}}$$

and one-sided differences at the endpoints. The result is an array of the **same length** as the input — one derivative value per data point.

**Key syntax:**
```python
dydx = np.gradient(y, x)   # always pass x when your spacing is not 1
```

### 15.2.1 Verification on a known function

Let's check `np.gradient` against the exact derivative of $\sin(x)$ (which is $\cos(x)$) before applying it to real data.

In [ ]:
x_sin = np.linspace(0, 2 * np.pi, 50)
y_sin = np.sin(x_sin)

# Numerical derivative
dy_num = np.gradient(y_sin, x_sin)

# Exact derivative
dy_exact = np.cos(x_sin)

max_error = np.max(np.abs(dy_num - dy_exact))
print(f"Max absolute error vs exact cos(x): {max_error:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(x_sin, y_sin, 'steelblue', linewidth=2, label='sin(x)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Original function: sin(x)')
ax.legend()

ax = axes[1]
ax.plot(x_sin, dy_exact, 'k-',  linewidth=2.5, label='Exact: cos(x)')
ax.plot(x_sin, dy_num,   'r--', linewidth=1.5, label='np.gradient (50 pts)')
ax.set_xlabel('x'); ax.set_ylabel("f '(x)")
ax.set_title('Derivative: numerical vs exact')
ax.legend()

plt.tight_layout()
plt.show()

### 15.2.2 Reaction rate from concentration data

In a batch reactor experiment, $C_A$ is measured at discrete time points. The rate of disappearance of A is:
$$r_A = -\frac{dC_A}{dt}$$

We only have a table of numbers — no formula — so `np.gradient` is the right tool.

In [ ]:
t   = np.array([0,    10,   20,   30,   40,   50,   60  ])  # s
C_A = np.array([1.00, 0.78, 0.61, 0.48, 0.37, 0.29, 0.23])  # mol/L

# Reaction rate: r_A = -dC_A/dt
# IMPORTANT: pass t as the second argument — spacing is 10 s, not 1
r_A = -np.gradient(C_A, t)

print(f"{'t (s)':>6}  {'C_A (mol/L)':>12}  {'r_A (mol/L·s)':>14}")
print("-" * 38)
for i in range(len(t)):
    print(f"  {t[i]:>4.0f}  {C_A[i]:>12.3f}  {r_A[i]:>14.5f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(t, C_A, 'o-', color='steelblue', linewidth=2, markersize=7)
ax.set_xlabel('Time (s)'); ax.set_ylabel('$C_A$ (mol/L)')
ax.set_title('Concentration vs. time')

ax = axes[1]
ax.plot(t, r_A, 's-', color='darkorange', linewidth=2, markersize=7)
ax.set_xlabel('Time (s)'); ax.set_ylabel('$r_A$ (mol/L·s)')
ax.set_title('Reaction rate vs. time')

plt.tight_layout()
plt.show()

(15.3)=
## 15.3 Numerical Integration of Discrete Data: Trapezoidal Rule

When you have a table of $(x, y)$ values and want $\int y \, dx$, the simplest method is the **trapezoidal rule**: connect neighboring points with straight lines and sum the trapezoid areas.

$$\int_a^b f(x)\,dx \approx \sum_{i=0}^{n-2} \frac{x_{i+1} - x_i}{2}\bigl(f_i + f_{i+1}\bigr)$$

`scipy.integrate.trapezoid(y, x)` implements this in one line.

### 15.3.1 Building intuition: visualizing trapezoids

Let's integrate $f(x) = \frac{4}{1+x^2}$ from 0 to 1, whose exact answer is $\pi$. We'll use only 5 points so the trapezoids are visible.

In [ ]:
def f_pi(x):
    return 4.0 / (1.0 + x**2)

# 5-point trapezoidal approximation
x_trap = np.linspace(0, 1, 5)
y_trap = f_pi(x_trap)

area_trap = trapezoid(y_trap, x_trap)

print(f"5-point trapezoidal estimate: {area_trap:.6f}")
print(f"True value (π):               {np.pi:.6f}")
print(f"Error:                        {abs(area_trap - np.pi):.6f}")

# Visualize
x_fine = np.linspace(0, 1, 300)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_fine, f_pi(x_fine), 'steelblue', linewidth=2.5, label='f(x) = 4/(1+x²)', zorder=3)
for i in range(len(x_trap) - 1):
    ax.fill_between([x_trap[i], x_trap[i+1]],
                    [y_trap[i], y_trap[i+1]], alpha=0.25, color='steelblue')
    ax.plot([x_trap[i], x_trap[i]], [0, y_trap[i]], 'steelblue', linewidth=0.8, alpha=0.5)
ax.plot([x_trap[-1], x_trap[-1]], [0, y_trap[-1]], 'steelblue', linewidth=0.8, alpha=0.5)
ax.scatter(x_trap, y_trap, color='red', s=50, zorder=5, label='Data points')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title(f'Trapezoidal rule (5 pts): estimate = {area_trap:.4f},  π = {np.pi:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

### 15.3.2 Heat required from tabulated $C_p$ data

A steam process requires heating from 400 K to 1000 K. Given tabulated heat capacity data:
$$\Delta H = \int_{400}^{1000} C_p(T) \, dT$$

In [ ]:
# Tabulated Cp data for steam (J/mol·K) — 7 measurement points
T_data  = np.array([400,  500,  600,  700,  800,  900,  1000])  # K
Cp_data = np.array([34.3, 35.2, 36.3, 37.5, 38.6, 39.7,  40.8])  # J/mol·K

delta_H = trapezoid(Cp_data, T_data)   # J/mol

print(f"ΔH = ∫Cp dT  =  {delta_H:.1f} J/mol  =  {delta_H/1000:.3f} kJ/mol")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(T_data, Cp_data, 'o-', color='darkorange', linewidth=2, markersize=8)
ax.fill_between(T_data, Cp_data, alpha=0.2, color='darkorange',
                label=f'ΔH = {delta_H/1000:.2f} kJ/mol')
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('$C_p$ (J/mol·K)')
ax.set_title('Heat capacity of steam')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

```{note}
Always pass `x` as the second argument: `trapezoid(y, x)`. If you write `trapezoid(y)` alone, it assumes unit spacing ($\Delta x = 1$) and gives a wrong answer when your x-axis has physical units like seconds or Kelvin.
```

(15.4)=
## 15.4 Integration of a Python Function: `scipy.integrate.quad`

When you have a **Python function** (not just data), use `quad` instead of `trapezoid`. It is adaptive — it automatically samples more points where the function changes rapidly — and achieves near machine-precision accuracy.

```python
from scipy.integrate import quad

result, error = quad(f, a, b)
```

`quad` always returns **two values**: the integral estimate and an error bound. Always unpack both.

### 15.4.1 Basic usage

In [ ]:
# Integrate f(x) = x^2 from 0 to 3  (exact answer = 9)
def f_sq(x):
    return x**2

result, error = quad(f_sq, 0, 3)

print(f"∫₀³ x² dx  =  {result:.10f}")
print(f"Exact answer:   9.0000000000")
print(f"Error estimate: {error:.2e}")

### 15.4.2 The motivation problem solved: $\Delta H$ for N₂

Now let's answer the opening question. The NIST Shomate equation for N₂ (298–6000 K) is:

$$C_p(T) = A + Bt + Ct^2 + Dt^3 + \frac{E}{t^2}, \quad t = T/1000 \quad [\text{J/mol·K}]$$

We integrate it from 300 K to 1200 K using both `quad` and `trapezoid` to compare their accuracy.

In [ ]:
# NIST Shomate coefficients for N2 (298-6000 K range)
A, B, C, D, E = 26.09200, 8.218801, -1.976141, 0.159274, 0.044434

def Cp_N2(T):
    t = T / 1000.0
    return A + B*t + C*t**2 + D*t**3 + E/t**2   # J/mol·K

T_low, T_high = 300.0, 1200.0

# --- Method 1: quad (high accuracy) ---
dH_quad, err = quad(Cp_N2, T_low, T_high)

# --- Method 2: trapezoid with only 10 points ---
T_10  = np.linspace(T_low, T_high, 10)
dH_trap10 = trapezoid(Cp_N2(T_10), T_10)

# --- Method 3: trapezoid with 100 points ---
T_100 = np.linspace(T_low, T_high, 100)
dH_trap100 = trapezoid(Cp_N2(T_100), T_100)

print(f"ΔH from 300 K to 1200 K:")
print(f"  quad          : {dH_quad:.4f} J/mol  (error bound: {err:.2e})")
print(f"  trapezoid 10  : {dH_trap10:.4f} J/mol  (error vs quad: {abs(dH_trap10-dH_quad):.4f})")
print(f"  trapezoid 100 : {dH_trap100:.4f} J/mol  (error vs quad: {abs(dH_trap100-dH_quad):.6f})")

# Plot Cp(T)
T_plot = np.linspace(T_low, T_high, 300)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(T_plot, Cp_N2(T_plot), 'steelblue', linewidth=2.5)
ax.fill_between(T_plot, Cp_N2(T_plot), alpha=0.15, color='steelblue',
                label=f'ΔH = {dH_quad/1000:.2f} kJ/mol')
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('$C_p$ (J/mol·K)')
ax.set_title('Heat capacity of N₂  (NIST Shomate equation)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 15.4.3 Passing extra parameters to `quad`

If your function has parameters, pass them with `args=`:

```python
result, _ = quad(Cp_N2_general, T_low, T_high, args=(A, B, C, D, E))
```

Or use a `lambda` to "bake in" the parameters:

```python
result, _ = quad(lambda T: Cp_N2_general(T, A, B, C, D, E), T_low, T_high)
```

Both are equivalent — choose whichever is clearer to you.

In [ ]:
def Cp_N2_general(T, A, B, C, D, E):
    t = T / 1000.0
    return A + B*t + C*t**2 + D*t**3 + E/t**2

# Method 1: args=
r1, _ = quad(Cp_N2_general, T_low, T_high, args=(A, B, C, D, E))

# Method 2: lambda
r2, _ = quad(lambda T: Cp_N2_general(T, A, B, C, D, E), T_low, T_high)

print(f"args=   result: {r1:.4f} J/mol")
print(f"lambda  result: {r2:.4f} J/mol")

(15.5)=
## 15.5 Cumulative Integration: `cumulative_trapezoid`

Sometimes you want the **running total** — the integral value at every point, not just the final number. For example: enthalpy at every temperature, not just the endpoint.

`cumulative_trapezoid(y, x, initial=0)` returns an array of the same length as input, where each element is $\int_{x_0}^{x_i} y \, dx$.

Use `initial=0` to set the starting value at the first point.

In [ ]:
# Enthalpy of N2 as a function of temperature, relative to 298 K
T_range = np.linspace(298, 1500, 500)   # K
Cp_range = Cp_N2(T_range)              # J/mol·K

# Cumulative integral: ΔH(T) = ∫₂₉₈ᵀ Cp dT'
dH_cumulative = cumulative_trapezoid(Cp_range, T_range, initial=0)  # J/mol

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(T_range, Cp_range, color='steelblue', linewidth=2)
ax.set_xlabel('Temperature (K)'); ax.set_ylabel('$C_p$ (J/mol·K)')
ax.set_title('Heat capacity of N₂')

ax = axes[1]
ax.plot(T_range, dH_cumulative / 1000, color='darkorange', linewidth=2)
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('ΔH(T) (kJ/mol)')
ax.set_title('Cumulative enthalpy change from 298 K\n' + r'$\Delta H(T) = \int_{298}^{T} C_p\,dT\'$')
ax.axhline(0, color='gray', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

print(f"ΔH at T=1200 K: {dH_cumulative[np.argmin(np.abs(T_range-1200))]/1000:.2f} kJ/mol")

(15.6)=
## 15.6 Chemical Engineering Application: PFR Design

### Problem setup

The **plug flow reactor (PFR) design equation** relates reactor volume $V$ to conversion $X$:

$$V = F_{A0} \int_0^{X_f} \frac{dX}{-r_A(X)}$$

For a liquid-phase second-order reaction A → products with rate law $-r_A = k C_A^2 = k C_{A0}^2(1-X)^2$:

| Parameter | Value |
|-----------|-------|
| Rate constant $k$ | 0.05 L/mol·s |
| Inlet concentration $C_{A0}$ | 2.0 mol/L |
| Molar feed rate $F_{A0}$ | 5.0 mol/s |
| Target conversion $X_f$ | 80% |

**Goal:** Find the reactor volume needed to achieve 80% conversion.

### Part A: The Levenspiel plot

The integrand $1/(-r_A)$ is called the **Levenspiel integrand**. Plotting it vs. $X$ helps visualize how difficult the integration is — notice how it grows steeply as $X \to 1$.

In [ ]:
k    = 0.05    # L/mol·s
C_A0 = 2.0     # mol/L
F_A0 = 5.0     # mol/s
X_f  = 0.80    # target conversion

def rate(X):
    return k * C_A0**2 * (1 - X)**2   # -r_A  (mol/L·s)

def levenspiel(X):
    return 1.0 / rate(X)              # 1/(-r_A)

X_plot = np.linspace(0, 0.95, 300)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(X_plot, levenspiel(X_plot), 'steelblue', linewidth=2.5)
X_shade = np.linspace(0, X_f, 300)
ax.fill_between(X_shade, levenspiel(X_shade), alpha=0.2, color='steelblue',
                label=f'Area = V/F_{{A0}}  at X={X_f}')
ax.axvline(X_f, color='red', linestyle='--', linewidth=1.5, label=f'X_f = {X_f}')
ax.set_xlabel('Conversion X')
ax.set_ylabel(r'$1/(-r_A)$  (L·s/mol)')
ax.set_title('Levenspiel plot: area = reactor volume / F$_{A0}$')
ax.set_ylim(0, 80)
ax.legend()
plt.tight_layout()
plt.show()

### Part B: Numerical integration with `quad`

The area under the Levenspiel curve gives $V/F_{A0}$. For this second-order rate law there is also an exact analytical answer:
$$V_\text{exact} = \frac{F_{A0}}{k C_{A0}^2} \cdot \frac{X_f}{1-X_f}$$

Let's verify the numerical result against it.

In [ ]:
integral, err = quad(levenspiel, 0, X_f)
V_numerical = F_A0 * integral

# Analytical answer for 2nd-order: V = (F_A0 / k*C_A0^2) * X_f/(1-X_f)
V_exact = (F_A0 / (k * C_A0**2)) * (X_f / (1 - X_f))

print(f"PFR volume for X = {X_f:.0%} conversion:")
print(f"  Numerical (quad): V = {V_numerical:.4f} L")
print(f"  Analytical:       V = {V_exact:.4f} L")
print(f"  Relative error:   {abs(V_numerical - V_exact)/V_exact:.2e}")

### Part C: Reactor sizing chart — conversion vs. volume

Using `cumulative_trapezoid` on the Levenspiel integrand, we can build a full design chart that shows the required volume for *any* conversion target.

In [ ]:
X_arr  = np.linspace(0, 0.95, 500)
V_arr  = F_A0 * cumulative_trapezoid(levenspiel(X_arr), X_arr, initial=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(V_arr, X_arr, 'steelblue', linewidth=2.5)
ax.axhline(X_f,         color='red',  linestyle='--', linewidth=1.5, label=f'X = {X_f}')
ax.axvline(V_numerical, color='red',  linestyle='--', linewidth=1.5, label=f'V = {V_numerical:.1f} L')
ax.scatter([V_numerical], [X_f], color='red', s=80, zorder=5)
ax.set_xlabel('Reactor volume V (L)')
ax.set_ylabel('Conversion X')
ax.set_title('PFR design chart: conversion vs. volume')
ax.legend()
plt.tight_layout()
plt.show()

## Chapter 15 Summary

| Task | Tool | Key syntax |
|------|------|-----------|
| Derivative of discrete data | `np.gradient` | `np.gradient(y, x)` |
| Integral of discrete data | `scipy.integrate.trapezoid` | `trapezoid(y, x)` |
| Integral of a Python function | `scipy.integrate.quad` | `result, err = quad(f, a, b)` |
| Running cumulative integral | `scipy.integrate.cumulative_trapezoid` | `cumulative_trapezoid(y, x, initial=0)` |
| Pass parameters to `quad` | `args=` keyword | `quad(f, a, b, args=(p1, p2))` |

### When to use which tool

- **Have discrete data?** → `trapezoid` (or `np.gradient` for derivatives)
- **Have a Python function?** → `quad` for highest accuracy
- **Need the value at every point, not just the endpoint?** → `cumulative_trapezoid`
- **More points always helps** `trapezoid`, but `quad` is already near machine precision